# Lab 8: Programmatically Demonstrate a Binary Classifier Using Naive Bayes

## Objective
Implement and evaluate a binary classifier using the Naive Bayes algorithm on a custom dataset with Python.

## Theory
The Naive Bayes classifier applies Bayes' Theorem to compute the probability of a class given observed features, assuming conditional independence among features. In binary classification, the algorithm calculates posterior probabilities for the two classes and selects the class with the higher probability. Laplace smoothing avoids zero probabilities, and using log probabilities keeps the computation numerically stable.

In [1]:
import pandas as pd
import numpy as np

# Load dataset
base_dir = r"d:/Aayush_Acharya/7th Sem/DWDM/Lab 8"
df = pd.read_csv(f"{base_dir}/loan_application_data.csv")

def train_naive_bayes(df, target_col):
    total_docs = len(df)
    classes = df[target_col].unique()
    priors = {c: len(df[df[target_col] == c]) / total_docs for c in classes}
    features = [col for col in df.columns if col != target_col]
    model = {}

    for c in classes:
        model[c] = {}
        df_c = df[df[target_col] == c]
        for feat in features:
            values = df_c[feat].unique()
            model[c][feat] = {
                val: (len(df_c[df_c[feat] == val]) + 1) / (len(df_c) + len(df[feat].unique()))
                for val in df[feat].unique()
            }
    return priors, model, classes


def predict(priors, model, classes, sample):
    probs = {}
    for c in classes:
        prob = np.log(priors[c])
        for feat, val in sample.items():
            prob += np.log(model[c][feat].get(val, 1e-6))
        probs[c] = prob
    return max(probs, key=probs.get)

priors, model, classes = train_naive_bayes(df, 'Approved')

test_samples = df.drop(columns=['Approved']).to_dict(orient='records')
actuals = df['Approved'].tolist()
predictions = [predict(priors, model, classes, s) for s in test_samples]
accuracy = sum(1 for p, a in zip(predictions, actuals) if p == a) / len(actuals)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

new_application = {'Has_Credit_Card': 1, 'Employed': 1, 'High_Income': 0, 'Good_Credit_History': 1}
prediction = predict(priors, model, classes, new_application)
print(f"Prediction for new application {new_application}: {'Approved' if prediction == 1 else 'Not Approved'}")

Model Accuracy: 100.00%
Prediction for new application {'Has_Credit_Card': 1, 'Employed': 1, 'High_Income': 0, 'Good_Credit_History': 1}: Approved


## Discussion
A Naive Bayes classifier was trained on a loan approval dataset with binary applicant features. Class priors and feature likelihoods were computed with Laplace smoothing, and log probabilities were used to avoid underflow. The model predicted approval outcomes for all examples and also evaluated a new loan application.

## Conclusion
Naive Bayes proved to be a fast and effective binary classifier for this dataset. Despite the simplifying assumption of feature independence, the model is useful for quick classification tasks and can handle binary feature data naturally.